In [1]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML
from lib.matrix_dataset import MatrixDataset
from recommenders.lmf import LogisticMatrixFactorization

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    device = mps_device
    x = torch.ones(1, device=mps_device)
    print(x)
else:
    print ("MPS device not found.")

if device == torch.device("cuda"):
    dtype = torch.float32
    print("Using CUDA.")
elif device == torch.device("cpu"):
    dtype = torch.float64
    print("Using CPU.")
elif device == torch.device("mps"):
    dtype = torch.float32
    print("Using MPS.")

# device = "cpu"
# dtype = torch.float64

tensor([1.], device='mps:0')
Using MPS.


# Import data

In [3]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 1, "long_term": 1},
)
df

TypeError: load_all_tracks() got an unexpected keyword argument 'penality_factors'

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

## Setup the dataset

In [ ]:
df_matrix_mf = df.copy()
df_matrix_mf.loc[df_matrix_mf["type"] == "liked_track", "affinity"] = 0.5
df_matrix_mf.loc[df_matrix_mf["type"] == "playlist", "affinity"] = 0.3

used_types = ["top_track", "liked_track", "playlist"]
used_types = ["top_track"]
df_matrix_mf = df_matrix_mf[df["type"].isin(used_types)]
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]


,username,id,affinity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
0,jaslkh,75rqqKvzJCGv2oq9C4yFDt,1.00,0.630,0.374,0.2270,0.8610,0.000075,0.2940,0.3730,104.955,-9.007,196426,2023,63
1,jaslkh,2FYGZDfsAnNsrm1gVbyKnG,0.98,0.827,0.768,0.2650,0.7900,0.000024,0.4970,0.7340,99.988,-5.702,137533,2022,70
2,jaslkh,4kroNlz8BTfswE4M0i3YCh,0.96,0.583,0.297,0.4060,0.8790,0.000000,0.1270,0.2910,124.279,-11.273,162906,2022,61
3,jaslkh,2N3YZ075lq9z1ObaAiX6l1,0.94,0.651,0.327,0.4020,0.9280,0.000000,0.2250,0.5510,135.325,-10.070,89749,2022,54
4,jaslkh,2SiAcexM2p1yX6joESbehd,0.92,0.555,0.634,0.2730,0.1290,0.000002,0.1880,0.5550,170.228,-5.522,174044,2023,73
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12318,dany,4GRMPDD3V6rM2BlmRdYCUJ,0.10,0.491,0.699,0.0924,0.1620,0.864000,0.1050,0.0762,130.325,-10.500,214000,2022,41
12319,dany,17Xof0GRZZfS7ZgjUJ27pH,0.08,0.271,0.503,0.0314,0.7500,0.000102,0.0941,0.0742,127.960,-4.142,241821,2020,50
12320,dany,2Rd4eJ4KwXQQn2sMSToyUM,0.06,0.319,0.236,0.0318,0.8980,0.000016,0.2720,0.2540,162.351,-9.644,156091,2021,6
12321,dany,4fPBB44eDH71YohayI4eKV,0.04,0.630,0.908,0.0326,0.0238,0.592000,0.1160,0.9680,120.522,-2.420,189226,2006,75


In [ ]:
# df_matrix_mf = pd.read_csv(os.path.join(data_path.DATA_PATH, "artificial_data.csv"), index_col=0)

In [ ]:
df_matrix_mf["affinity"] *= 1

In [ ]:
matrix_mf = MatrixDataset(df_matrix_mf, "username", "id", "affinity", device=device, dtype=dtype)
R = matrix_mf.R
R.to(device)
R.shape

torch.Size([8, 923])

In [ ]:
alpha = matrix_mf.compute_alpha().item()
R *= alpha
alpha

10.527777671813965

In [ ]:
R

tensor([[0.0000, 0.0000, 6.5272,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 8.8433],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [4.6322, 6.3167, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 9.6856, 0.0000, 0.0000]],
       device='mps:0')

In [ ]:
num_latent_factors = 3
lmf = LogisticMatrixFactorization(
    counts=R,
    num_factors=num_latent_factors,
    reg_param=0.01,
    device=device,
    dtype=dtype,
)
lmf.to(device)

num_epochs = 100
lmf.train_model(
    num_epochs=num_epochs,
    learning_rate=1,
    log_interval=1,
    tqdm=False
)

Epoch: 1, log-likelihood: 8083.0674, MPR: 0.1860
Epoch: 2, log-likelihood: 7206.9624, MPR: 0.1969
Epoch: 3, log-likelihood: 5633.9463, MPR: 0.1064
Epoch: 4, log-likelihood: 4922.1699, MPR: 0.0939
Epoch: 5, log-likelihood: 4365.6475, MPR: 0.0840
Epoch: 6, log-likelihood: 4044.5720, MPR: 0.0754
Epoch: 7, log-likelihood: 3835.4778, MPR: 0.0706
Epoch: 8, log-likelihood: 3688.9722, MPR: 0.0676
Epoch: 9, log-likelihood: 3579.4253, MPR: 0.0642
Epoch: 10, log-likelihood: 3496.1040, MPR: 0.0626
Epoch: 11, log-likelihood: 3431.9084, MPR: 0.0611
Epoch: 12, log-likelihood: 3399.6892, MPR: 0.0606
Epoch: 13, log-likelihood: 3340.1677, MPR: 0.0593
Epoch: 14, log-likelihood: 3282.5676, MPR: 0.0581
Epoch: 15, log-likelihood: 3222.8381, MPR: 0.0581
Epoch: 16, log-likelihood: 3169.9709, MPR: 0.0567
Epoch: 17, log-likelihood: 3122.5405, MPR: 0.0574
Epoch: 18, log-likelihood: 3080.5771, MPR: 0.0556
Epoch: 19, log-likelihood: 3041.4014, MPR: 0.0563
Epoch: 20, log-likelihood: 3011.2158, MPR: 0.0550
Epoch: 21

In [ ]:
lmf.save("models", "lmf")

In [ ]:
px.line(x=range(len(lmf.losses)), y=lmf.losses.cpu(), title="Loss").show()
px.line(x=range(len(lmf.mpr_history)), y=lmf.mpr_history.cpu(), title="MPRS").show()

In [ ]:
user_id = matrix_mf.usernames_to_ids(["paul"])[0]
print(user_id)

# Get the top 10 recommendations for the user
top_10_ids, top_10_ids_scores = lmf.recommend(user_id, top_k=20, filter_user_items=True)

# Get the top 10 recommendations for the user
matrix_mf.item_ids_to_df(top_10_ids)

7


,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,1.00,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
11,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162929,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/7y...,https://api.spotify.com/v1/tracks/7y1r8U18JSN9...,7y1r8U18JSN93zZARYKfhT,...,medium_term,0.78,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
12,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,194543,False,{'isrc': 'USSM12301258'},{'spotify': 'https://open.spotify.com/track/60...,https://api.spotify.com/v1/tracks/609E1JCInJnc...,609E1JCInJncactoMmkDon,...,medium_term,0.76,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
18,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,127733,False,{'isrc': 'GBCBS6732267'},{'spotify': 'https://open.spotify.com/track/56...,https://api.spotify.com/v1/tracks/560OsnDa7tps...,560OsnDa7tpsK2B0krVlOP,...,medium_term,0.64,jaslkh,1968,"[pop, english, rock, rock, rock]",NaT,NaN,NaN,NaN,NaN
21,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,272533,False,{'isrc': 'USIR10311862'},{'spotify': 'https://open.spotify.com/track/0x...,https://api.spotify.com/v1/tracks/0xmjwnQ3FNE6...,0xmjwnQ3FNE6HuWCt2nHdZ,...,medium_term,0.58,jaslkh,2003,"[pop, pop, pop]",NaT,NaN,NaN,NaN,NaN
35,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,179379,True,{'isrc': 'USUG12203930'},{'spotify': 'https://open.spotify.com/track/70...,https://api.spotify.com/v1/tracks/70msjvZ6xw9e...,70msjvZ6xw9eGcjBPI5osG,...,medium_term,0.30,jaslkh,2022,"[rnb, pop]",NaT,NaN,NaN,NaN,NaN
40,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,194087,False,{'isrc': 'USUM71900764'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2Fxmhks0bxGS...,2Fxmhks0bxGSBdJ92vM42m,...,medium_term,0.20,jaslkh,2019,"[pop, pop]",NaT,NaN,NaN,NaN,NaN
54,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,206728,True,{'isrc': 'USUG12203928'},{'spotify': 'https://open.spotify.com/track/6P...,https://api.spotify.com/v1/tracks/6PFyd6Gmp2LZ...,6PFyd6Gmp2LZHI1IpkZGgy,...,short_term,0.92,jaslkh,2022,"[rnb, pop]",NaT,NaN,NaN,NaN,NaN
55,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,179379,True,{'isrc': 'USUG12203930'},{'spotify': 'https://open.spotify.com/track/70...,https://api.spotify.com/v1/tracks/70msjvZ6xw9e...,70msjvZ6xw9eGcjBPI5osG,...,short_term,0.90,jaslkh,2022,"[rnb, pop]",NaT,NaN,NaN,NaN,NaN
57,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,272533,False,{'isrc': 'USIR10311862'},{'spotify': 'https://open.spotify.com/track/0x...,https://api.spotify.com

In [ ]:
if lmf.num_factors <= 3:
    df_tracks = df_matrix_mf.copy()
    df_tracks = df_tracks.sample(frac=1) # Shuffle the dataframe
    df_tracks = df_tracks[:150] # Keep only 1000 tracks

    # Add item latent factors to the dataframe
    items_latent_columns = [f"track_latent_{i}" for i in range(lmf.num_factors)]
    for track_id in df_tracks["id"].unique():
        latent_factors = lmf.get_item_factors([matrix_mf.itemnames_to_ids([track_id])[0]]).tolist()
        df_tracks.loc[df_tracks["id"] == track_id, items_latent_columns] = latent_factors

    # Add user latent factors to the dataframe
    users_latent_columns = [f"latent_factor_{i}" for i in range(lmf.num_factors)]
    for user_id in df_tracks["username"].unique():
        latent_factors = lmf.get_user_factors([matrix_mf.usernames_to_ids([user_id])[0]]).tolist()
        df_tracks.loc[df_tracks["username"] == user_id, users_latent_columns] = latent_factors

    plotting.plot_latent_space(
        df=df_tracks,
        color=df_tracks["username"],
        text=df_tracks["username"],
        latent_columns=items_latent_columns,
    ).show()

    plotting.plot_latent_space(
        df=df_tracks,
        color=df_tracks["username"],
        text=df_tracks["username"],
        latent_columns=users_latent_columns,
    ).show()